# Seminário 2: Implementação do pipeline multimodal de extração de features

Este notebook apresenta a prova de conceito (PoC) da arquitetura de extração de dados inspirada no artigo *"Decoding reddit memes virality"*.

**Contexto de big data:** Devido à alta dimensionalidade dos dados (milhares de imagens e modelos pesados de *deep learning*), processar o *dataset* completo sequencialmente na nuvem pública é inviável. Portanto, este notebook demonstra a **lógica de extração** aplicada a uma pequena amostra. No ambiente de produção do grupo, essas mesmas funções são encapsuladas como *User Defined Functions (UDFs)* e paralelizadas utilizando Apache Spark sobre um *cluster* com múltiplas GPUs.

As etapas demonstradas aqui incluem:
* **Visão computacional:** detecção de objetos (YOLOv5), análise de emoções faciais (FER) e estatísticas de cor.
* **Processamento de Linguagem Natural (PLN):** reconhecimento óptico de caracteres (EasyOCR) e extração de semântica textual (Sentence-Transformers).

## 1 — Provisionamento do ambiente
Instalação das bibliotecas necessárias para a execução dos modelos de visão computacional (YOLOv5, OpenCV), processamento de linguagem natural (Sentence-Transformers) e ferramentas de persistência de dados em larga escala (PyArrow/Pandas).

In [ ]:
# Execute esta célula no ambiente Colab. Num ambiente local, algumas destas instalações podem ser omitidas.
!pip install -q git+https://github.com/ultralytics/yolov5.git@master # yolov5
!pip install -q easyocr fer sentence-transformers opencv-python-headless pillow pandas pyarrow tqdm

## 2 — Aceleração de hardware e inicialização do OCR
Verificação da disponibilidade de tensores na GPU (CUDA) para evitar gargalos de processamento (*bottlenecks*). Além disso, instanciamos o modelo do `EasyOCR` no padrão *Singleton*, garantindo que ele seja carregado na memória apenas uma vez, uma prática essencial para otimizar o processamento em lote.

In [ ]:
import torch
print('CUDA disponível:', torch.cuda.is_available())
print('Versão do Torch:', torch.__version__)

import os, sys, json, time
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm import tqdm
import cv2
from PIL import Image

# Wrapper para o EasyOCR, reaproveitando o padrão arquitetural do repositório original
import easyocr
_reader = None
def get_reader(model_storage_directory='/content/easyocr_models', gpu=True):
    global _reader
    if _reader is None:
        _reader = easyocr.Reader(['en'], gpu=gpu, download_enabled=True, model_storage_directory=model_storage_directory, verbose=False)
    return _reader

def run_easyocr_on_path(path, detail=1):
    reader = get_reader()
    return reader.readtext(path, detail=detail)

## 3 — Carregamento dos modelos de IA (visão e linguagem)
Nesta etapa, alocamos em memória os dois modelos principais que guiarão a extração de *features* complexas:
1. **YOLOv5:** Uma arquitetura leve e extremamente rápida para detecção de múltiplos objetos nas imagens.
2. **Sentence-Transformers (MiniLM):** Um modelo baseado em *Transformers* (uma variação mais eficiente do BERT) que converte o texto extraído do meme em vetores densos bidimensionais (*embeddings*), permitindo agrupamentos semânticos no futuro.

In [ ]:
import torch

# Carregamento do YOLOv5 através de importação direta
from yolov5 import load as yolo_load
model_yolo = yolo_load('yolov5s', pretrained=True)
model_yolo.conf = 0.25  # Limiar de confiança (confidence threshold)

# Sentence-Transformers para a geração de embeddings de texto
from sentence_transformers import SentenceTransformer
model_text = SentenceTransformer('all-MiniLM-L6-v2', device='cuda' if torch.cuda.is_available() else 'cpu')
print('Modelos carregados com sucesso')

## 4 — Arquitetura de extração (pipeline core)
Este é o núcleo da transformação dos dados não estruturados em dados estruturados. Para cada imagem lida, o sistema:
1. Detecta e recorta os objetos de interesse (`run_yolov5`).
2. Extrai médias e desvios de propriedades de cor em RGB e HSV (`color_stats_from_crop`).
3. Tenta identificar rostos humanos e classifica a emoção dominante (`run_fer_on_crop`).
4. Extrai o texto contido na imagem e o transforma em *embeddings* de significado (`run_easyocr_on_crop` e `text_to_embedding`).

In [ ]:
from fer import FER
face_detector = FER(mtcnn=True)

def run_yolov5(image_path_or_array):
    # Aceita o caminho do ficheiro ou uma matriz (array) NumPy no formato BGR
    if isinstance(image_path_or_array, (str,)):
        results = model_yolo(image_path_or_array)
        df = results.pandas().xyxy[0]
        img = cv2.imread(image_path_or_array)
    else:
        results = model_yolo(image_path_or_array)
        df = results.pandas().xyxy[0]
        img = image_path_or_array.copy()

    records = []
    for i, row in df.iterrows():
        x1, y1, x2, y2 = int(row.xmin), int(row.ymin), int(row.xmax), int(row.ymax)
        crop = img[y1:y2, x1:x2] if y2>y1 and x2>x1 else None
        records.append({
            'class': row.name,
            'label': row['name'],
            'conf': float(row.confidence),
            'bbox': (x1,y1,x2,y2),
            'crop': crop
        })
    return records

def color_stats_from_crop(crop):
    if crop is None:
        return {}
    # Recorte em formato BGR (OpenCV) -> conversão para o formato RGB
    rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
    r,g,b = cv2.split(rgb)
    stats = {
        'r_mean': float(r.mean()), 'g_mean': float(g.mean()), 'b_mean': float(b.mean()),
        'r_std': float(r.std()), 'g_std': float(g.std()), 'b_std': float(b.std())
    }
    hsv = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV)
    h,s,v = cv2.split(hsv)
    stats.update({'h_mean': float(h.mean()), 's_mean': float(s.mean()), 'v_mean': float(v.mean())})
    return stats

def run_fer_on_crop(crop):
    # O modelo FER requer entradas no formato RGB (PIL ou NumPy). Conversão BGR -> RGB
    if crop is None:
        return {}
    rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
    # O método face_detector.detect_emotions aceita matrizes NumPy em RGB
    try:
        faces = face_detector.detect_emotions(rgb)
        if not faces:
            return {}
        # Utilizar a primeira face detetada dentro da área recortada
        top = faces[0]
        emotions = top.get('emotions', {})
        dominant = max(emotions.items(), key=lambda x: x[1])[0] if emotions else None
        return {'emotions': emotions, 'dominant_emotion': dominant}
    except Exception as e:
        return {'error': str(e)}

def run_easyocr_on_crop(crop):
    # A função readtext do EasyOCR aceita um caminho ou uma matriz RGB. Conversão BGR -> RGB para aplicar o reader.readtext
    if crop is None:
        return []
    rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
    results = get_reader().readtext(rgb, detail=1)
    return results

def text_to_embedding(text):
    if text is None or text.strip()=='' :
        return np.zeros((model_text.get_sentence_embedding_dimension(),), dtype=np.float32)
    emb = model_text.encode(text, convert_to_numpy=True, show_progress_bar=False)
    return emb.astype(np.float32)

def process_image(image_path):
    recs = []
    img = cv2.imread(image_path)
    detections = run_yolov5(image_path)

    # Aplicação do OCR à imagem na sua totalidade (global)
    ocr_global = run_easyocr_on_path(image_path)
    ocr_global_text = ' '.join([r[1] for r in ocr_global]) if ocr_global else ''
    emb_global = text_to_embedding(ocr_global_text)

    for i, d in enumerate(detections):
        crop = d['crop']
        cstats = color_stats_from_crop(crop)
        fer_res = run_fer_on_crop(crop)
        ocr_crop = run_easyocr_on_crop(crop)
        ocr_text = ' '.join([r[1] for r in ocr_crop]) if ocr_crop else ''
        emb = text_to_embedding(ocr_text if ocr_text else ocr_global_text)

        record = {
            'image_path': image_path,
            'detection_id': i,
            'label': d['label'],
            'conf': d['conf'],
            'bbox': d['bbox'],
            'ocr_text': ocr_text,
            'ocr_global_text': ocr_global_text,
            'embedding': emb,
        }
        record.update(cstats)
        record.update(fer_res if isinstance(fer_res, dict) else {})
        recs.append(record)

    # Caso não existam deteções, regista-se na mesma uma entrada com o resultado do OCR global
    if not detections:
        recs.append({
            'image_path': image_path, 'detection_id': -1, 'label': None, 'conf': None, 'bbox': None,
            'ocr_text': '', 'ocr_global_text': ocr_global_text, 'embedding': emb_global
        })
    return recs

## 5 — Execução em lote e persistência de dados
Demonstração do *pipeline* rodando sobre uma amostra de até 50 imagens. Ao final da execução, os dados são convertidos de memória para disco utilizando Apache Parquet.

A escolha do formato Parquet é fundamental em arquiteturas de *big data* por ser colunar, altamente comprimido e otimizado para leituras analíticas subsequentes. Já as matrizes de *embeddings* textuais são salvas no formato otimizado da biblioteca NumPy (`.npz`).

In [ ]:
IMAGES_DIR = '/content/images'  # Ajustar o caminho caso se utilize o Google Drive
os.makedirs(IMAGES_DIR, exist_ok=True)

# Compilação da lista de ficheiros de imagens
import glob
images = glob.glob(f'{IMAGES_DIR}/**/*.jpg', recursive=True) + glob.glob(f'{IMAGES_DIR}/**/*.png', recursive=True)
print('Encontradas', len(images), 'imagens')

sample = images[:50]  # Processar um limite padrão de 50 imagens para a prova de conceito
all_records = []
for p in tqdm(sample):
    try:
        recs = process_image(p)
        all_records.extend(recs)
    except Exception as e:
        print('Erro em', p, e)

# Conversão dos resultados para DataFrame e respetiva persistência
if all_records:
    df = pd.DataFrame([{k:v for k,v in r.items() if k!='embedding'} for r in all_records])
    # Guardar as representações vetoriais (embeddings) em separado, mantendo o alinhamento através do índice
    embs = np.stack([r['embedding'] for r in all_records])
    df.to_parquet('/content/feature_table.parquet', index=False)
    np.savez_compressed('/content/embeddings.npz', embeddings=embs)
    print('Guardado /content/feature_table.parquet e /content/embeddings.npz com sucesso')
else:
    print('Nenhum registo gerado')

## Próximos passos
Com o arquivo `feature_table.parquet` gerado e salvo na máquina/servidor, o próximo passo consiste em utilizar essas *features* tabulares (junto aos *embeddings*) para treinar modelos preditivos, como Regressão Logística ou CatBoost, a fim de prever estatisticamente a probabilidade de viralização de um meme.